In [0]:
# Import required PySpark functions
from pyspark.sql.functions import (
    current_timestamp, regexp_replace, trim, col, when, lit,
    row_number, monotonically_increasing_id, max as spark_max, 
    first, last, sum
)
from pyspark.sql.window import Window
from pyspark.sql.types import BooleanType, DoubleType, IntegerType

print("✓ All imports loaded successfully")

In [0]:
# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS 02_silver.clean")

print("✓ Silver schema '02_silver.clean' ready")
print("  Tables will be written to: 02_silver.clean.<table_name>")

In [0]:
def clean_bronze_to_silver(table_name):

        # Read from bronze
    df = spark.table(f"01_bronze.raw.{table_name}")
    initial_count = df.count()
    print(f"Initial row count: {initial_count:,}")
    
    # STEP 1: Standardize column names to snake_case
    df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])
    print("✓ Standardized column names to snake_case")
    
    # STEP 2: Trim spaces from all string columns
    for column in df.columns:
        df = df.withColumn(column, trim(col(column)))
    print("✓ Trimmed whitespace from all columns")
    
    # STEP 3: Convert empty strings to NULL
    for column in df.columns:
        df = df.withColumn(column, 
            when((col(column) == "") | (col(column) == " "), lit(None))
            .otherwise(col(column)))
    print("✓ Converted empty strings to NULL")
    
    # STEP 4: Remove duplicate rows ONLY (keep all other rows)
    before_dedup = df.count()
    df = df.dropDuplicates()
    after_dedup = df.count()
    if before_dedup != after_dedup:
        print(f"✓ Removed {before_dedup - after_dedup} duplicate rows")
    else:
        print("✓ No duplicate rows found")
    
    # STEP 5: Fill NULL IDs sequentially based on row position
    id_column = f"{table_name}_id"
    
    if id_column in df.columns:
        null_count = df.filter(col(id_column).isNull()).count()
        if null_count > 0:
            print(f"  Found {null_count} null {id_column} values - filling sequentially...")
            
            df = df.withColumn("_original_order", monotonically_increasing_id())
            
            window_spec = Window.orderBy("_original_order").rowsBetween(Window.unboundedPreceding, 0)
            
            df = df.withColumn("_last_valid_id", 
                last(col(id_column), ignorenulls=True).over(window_spec))
            
            df = df.withColumn("_null_count",
                when(col(id_column).isNull(), 1).otherwise(0))
           
            df = df.withColumn("_cumsum_nulls",
                when(col(id_column).isNotNull(), 0)
                .otherwise(sum(col("_null_count")).over(window_spec)))
            
            df = df.withColumn(id_column,
                when(col(id_column).isNull(),
                    when(col("_last_valid_id").isNotNull(),
                         (col("_last_valid_id").cast("int") + col("_cumsum_nulls")).cast("string"))
                    .otherwise((col("_cumsum_nulls")).cast("string")))
                .otherwise(col(id_column)))
        
            df = df.drop("_original_order", "_last_valid_id", "_null_count", "_cumsum_nulls")
            print(f"  ✓ Filled {null_count} null {id_column} values sequentially (prev_id + 1)")
    
    # STEP 6: Convert is_active columns to boolean
    for column in df.columns:
        if "is_active" in column.lower():
            print(f"  Converting {column} to boolean...")
            df = df.withColumn(column,
                when(col(column).isin("1", "Yes", "Y", "true", "True", "TRUE"), True)
                .when(col(column).isin("0", "No", "N", "false", "False", "FALSE"), False)
                .otherwise(None).cast(BooleanType()))
            print(f"  ✓ Converted {column} to boolean")
    
    # STEP 7: Table-specific cleaning
    if table_name == "opportunity":
        if "revenue_amount" in df.columns:
            print("  Standardizing revenue_amount...")
            df = df.withColumn("revenue_amount", 
                regexp_replace(col("revenue_amount"), "[^0-9.\\-]", ""))
           
            df = df.withColumn("revenue_amount", 
                when((col("revenue_amount").isNotNull()) & (col("revenue_amount") != ""),
                     col("revenue_amount").cast(DoubleType()))
                .otherwise(lit(0.0)))
            print("  ✓ Standardized revenue_amount to numeric (removed all currency symbols)")
    
    if table_name == "product":
        if "list_price" in df.columns:
       
            df = df.withColumn("list_price",
                when((col("list_price").isNotNull()) & (col("list_price") != ""),
                     col("list_price").cast(DoubleType()))
                .otherwise(lit(0.0)))
            print("  ✓ Converted list_price to numeric")
    
    # STEP 8: Add transformation metadata
    df = df.withColumn("processed_time", current_timestamp())
    
    # STEP 9: Write to silver with schema overwrite
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"02_silver.clean.{table_name}")
    
    final_count = df.count()
    print(f"\n✓ {table_name} cleaned successfully")
    print(f"  Final row count: {final_count:,}")
    print(f"  Rows preserved: {final_count} (100% - NO ROWS REMOVED)")
    
    return df

print("✓ Cleaning function defined")

In [0]:

df_emp = spark.table("02_silver.clean.employee")
total_rows = df_emp.count()
print(f"\nTotal rows: {total_rows}")

# Show ALL employee rows to verify cleaning
print(f"\nShowing ALL {total_rows} employee rows with cleaned data:")
df_emp_all = df_emp.select("employee_id", "employee_name", "role", "region", "is_active_flag") \
    .orderBy(col("employee_id").cast("int"))

display(df_emp_all)


In [0]:

df_opp = spark.table("02_silver.clean.opportunity")
total_rows = df_opp.count()
print(f"\nTotal rows: {total_rows:,}")
print("  - All opportunity_id filled (no nulls)")
print("  - revenue_amount standardized (numeric, no currency symbols)")
print("\nTo see ALL rows: Remove .limit({SAMPLE_SIZE}) from the code below")
print("")

df_opp_sample = df_opp.select("opportunity_id", "customer_id", "product_id", "employee_id", "revenue_amount", "close_status", "contract_term", "start_date", "end_date") \
    .orderBy(col("opportunity_id").cast("int")) \
    .limit(150000)
display(df_opp_sample)

print("✓ All revenue_amount values standardized (removed £, $, €, commas)")


In [0]:

df_prod = spark.table("02_silver.clean.product")
print(f"\nTotal rows: {df_prod.count()}")

df_prod_all = df_prod.select("product_id", "product_name", "plan_name", "billing_cycle", "list_price", "is_active") \
    .orderBy("product_id")

display(df_prod_all)

print("✓ All 2 null product_id values have been filled sequentially")
print("✓ list_price converted to numeric (double)")
print("✓ is_active converted to boolean (True/False)")

In [0]:
df_cust = spark.table("02_silver.clean.customer")
total_rows = df_cust.count()
print(f"\nTotal rows: {total_rows:,}")

# Show ALL customer rows to verify cleaning
print(f"\nShowing ALL {total_rows:,} customer rows with cleaned data:")
df_cust_all = df_cust.select("customer_id", "customer_name", "country", "industry_type", "account_created_date", "is_active") \
    .orderBy(col("customer_id").cast("int"))

display(df_cust_all)


In [0]:

df_fx = spark.table("02_silver.clean.fx_rate")
total_rows = df_fx.count()
print(f"\nTotal rows: {total_rows}")

# Show ALL fx_rate rows
print(f"\nShowing ALL {total_rows} fx_rate rows with cleaned data:")
df_fx_all = df_fx.select("*") \
    .orderBy("currency_code")

display(df_fx_all)


In [0]:

# Read from BRONZE table (original data) to get proper format
df_customer = spark.table("01_bronze.raw.customer")

print("\nBefore conversion:")
for field in df_customer.schema.fields:
    if field.name in ["customer_id", "Account Created Date"]:
        print(f"  - {field.name}: {field.dataType}")

# Standardize column names first
df_customer = df_customer.toDF(*[c.lower().replace(" ", "_") for c in df_customer.columns])

# Convert datatypes - dates are in DD-MM-YYYY format
df_customer = df_customer \
    .withColumn("customer_id", col("customer_id").cast(IntegerType())) \
    .withColumn("account_created_date", to_date(col("account_created_date"), "dd-MM-yyyy"))

print("\nAfter conversion:")
for field in df_customer.schema.fields:
    if field.name in ["customer_id", "account_created_date"]:
        print(f"  - {field.name}: {field.dataType}")

# Write back to silver table
df_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.customer")

print("\n✓ Customer table datatypes updated successfully")

In [0]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType, DateType


# Read from BRONZE table (original data)
df_employee = spark.table("01_bronze.raw.employee")

for field in df_employee.schema.fields:
    if field.name in ["employee_id", "Hire Date", "Last Update"]:
        print(f"  - {field.name}: {field.dataType}")

# Standardize column names first
df_employee = df_employee.toDF(*[c.lower().replace(" ", "_") for c in df_employee.columns])

before_filter = df_employee.count()
df_employee = df_employee.filter(col("employee_id").isNotNull())
after_filter = df_employee.count()
removed_rows = before_filter - after_filter

# Convert datatypes - dates are in DD-MM-YYYY format
df_employee = df_employee \
    .withColumn("employee_id", col("employee_id").cast(IntegerType())) \
    .withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy")) \
    .withColumn("last_update", to_date(col("last_update"), "dd-MM-yyyy"))

df_employee = df_employee.withColumn(
    "is_active_flag",
    when(col("is_active_flag").isin("Y", "Yes", "1", "yes", "y", "true", "True", "TRUE"), True)
    .when(col("is_active_flag").isin("N", "No", "0", "no", "n", "false", "False", "FALSE"), False)
    .otherwise(None).cast(BooleanType())
)

print("\nAfter conversion:")
for field in df_employee.schema.fields:
    if field.name in ["employee_id", "hire_date", "last_update", "is_active_flag"]:
        print(f"  - {field.name}: {field.dataType}")

# Write back to silver table
df_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.employee")

print("\n✓ Employee table datatypes updated successfully")

In [0]:

from pyspark.sql.functions import col, to_date
from pyspark.sql.types import FloatType, DateType

# Read from BRONZE table
df_fx_rate = spark.table("01_bronze.raw.fx_rate")


for field in df_fx_rate.schema.fields:
    if field.name in ["FX Rate to GBP", "Effective Date"]:
        print(f"  - {field.name}: {field.dataType}")

# Standardize column names first
df_fx_rate = df_fx_rate.toDF(*[c.lower().replace(" ", "_") for c in df_fx_rate.columns])

df_fx_rate = df_fx_rate \
    .withColumn("fx_rate_to_gbp", col("fx_rate_to_gbp").cast(FloatType())) \
    .withColumn("effective_date", to_date(col("effective_date"), "yyyy-MM-dd"))

print("\nAfter conversion:")
for field in df_fx_rate.schema.fields:
    if field.name in ["fx_rate_to_gbp", "effective_date"]:
        print(f"  - {field.name}: {field.dataType}")

# Write back to silver table
df_fx_rate.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.fx_rate")

print("\n✓ FX Rate table datatypes updated successfully")

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, regexp_replace
from pyspark.sql.types import IntegerType, DateType, TimestampType, DoubleType

# Read from BRONZE table
df_opportunity = spark.table("01_bronze.raw.opportunity")

print(f"\nInitial rows from bronze: {df_opportunity.count():,}")

print("\nBefore conversion:")
for field in df_opportunity.schema.fields:
    if field.name in ["opportunity_id", "customer_id", "product_id", "employee_id", "Start Date", "End Date", "Created Timestamp", "Revenue Amount"]:
        print(f"  - {field.name}: {field.dataType}")
df_opportunity = df_opportunity.toDF(*[c.lower().replace(" ", "_") for c in df_opportunity.columns])

print("\nRemoving rows with NULL opportunity_id...")
before_filter = df_opportunity.count()
df_opportunity = df_opportunity.filter(col("opportunity_id").isNotNull())
after_filter = df_opportunity.count()
removed_rows = before_filter - after_filter
df_opportunity = df_opportunity \
    .withColumn("opportunity_id", col("opportunity_id").cast(IntegerType())) \
    .withColumn("customer_id", col("customer_id").cast(IntegerType())) \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("employee_id", col("employee_id").cast(IntegerType())) \
    .withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy")) \
    .withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy")) \
    .withColumn("created_timestamp", to_timestamp(col("created_timestamp"), "dd-MM-yyyy HH:mm"))

df_opportunity = df_opportunity \
    .withColumn("revenue_amount", regexp_replace(col("revenue_amount"), "[^0-9.\\-]", "")) \
    .withColumn("revenue_amount", col("revenue_amount").cast(DoubleType()))
for field in df_opportunity.schema.fields:
    if field.name in ["opportunity_id", "customer_id", "product_id", "employee_id", "start_date", "end_date", "created_timestamp", "revenue_amount"]:
        print(f"  - {field.name}: {field.dataType}")

# Write back to silver table
df_opportunity.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.opportunity")

print("\n✓ Opportunity table datatypes updated successfully")

In [0]:
from pyspark.sql.functions import col, to_date, when
from pyspark.sql.types import IntegerType, DateType, BooleanType, DoubleType

# Read from BRONZE table
df_product = spark.table("01_bronze.raw.product")

print(f"\nInitial rows from bronze: {df_product.count()}")

print("\nBefore conversion:")
for field in df_product.schema.fields:
    if field.name in ["product_id", "Created Date", "Is Active", "List Price"]:
        print(f"  - {field.name}: {field.dataType}")

df_product = df_product.toDF(*[c.lower().replace(" ", "_") for c in df_product.columns])

print("\nRemoving rows with NULL product_id...")
before_filter = df_product.count()
df_product = df_product.filter(col("product_id").isNotNull())
after_filter = df_product.count()
removed_rows = before_filter - after_filter
print(f"  ✓ Removed {removed_rows} rows with NULL product_id")
print(f"  ✓ Remaining rows: {after_filter}")

# Convert datatypes
df_product = df_product \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("created_date", to_date(col("created_date"), "dd-MM-yyyy"))

df_product = df_product.withColumn(
    "is_active",
    when(col("is_active").isin("Y", "Yes", "1", "yes", "y", "true", "True", "TRUE"), True)
    .when(col("is_active").isin("N", "No", "0", "no", "n", "false", "False", "FALSE"), False)
    .otherwise(None).cast(BooleanType())
)
df_product = df_product.withColumn("list_price", col("list_price").cast(DoubleType()))
for field in df_product.schema.fields:
    if field.name in ["product_id", "created_date", "is_active", "list_price"]:
        print(f"  - {field.name}: {field.dataType}")

# Write back to silver table
df_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.product")

print("\n✓ Product table datatypes updated successfully")